In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import scipy as sp
import matplotlib.pyplot as plt

from mis_dro.dataset import sample_dgp
from mis_dro.bayes_conjugates import *
from mis_dro.likelihood import cholesky_param_to_covariance
from mis_dro.plot import *
from mis_dro.experiments import ExperimentName
from mis_dro.main import get_npl_uuid, POSTERIOR_GB_COLS

from matplotlib import rc
rc('font', **{'family': 'serif', 'serif': ['Computer Modern'], "size":20})
rc('text', usetex=True)

In [ ]:
experiment_name = ExperimentName.mmd_portfolio_synthetic
mmd_experiment_dir = Path(f"/Users/patrick/experiments/misdro/2025_01_30_robas/2025_01_30_{experiment_name.value}")
npl_samples_dir = Path(f"/Users/patrick/experiments/misdro/2025_01_30_robas/2025_01_30_{experiment_name.value}_samples")
assert mmd_experiment_dir.exists()
assert npl_samples_dir.exists()
with open(mmd_experiment_dir / "experiment.json", "r", encoding="utf-8") as json_file:
    experiment = json.load(json_file)
mmd_results_df = pd.read_csv(mmd_experiment_dir / "results.csv", index_col=["uuid", "replication"])

kl_experiment_dir = Path("/Users/patrick/experiments/misdro/2025_01_30_robas/2025_01_30_kl_portfolio_synthetic")
assert kl_experiment_dir.exists()
kl_results_df = pd.read_csv(kl_experiment_dir / "results.csv", index_col=["uuid", "replication"])

all_results_df = pd.concat([mmd_results_df, kl_results_df])
all_results_df

In [ ]:
# NOTE filter by DGP
filter_dgp = "portfolio_contaminated_multivariate_normal"
dataset = "portfolio_synthetic"
# assert filter_dgp in all_results_df["dgp"] and dataset in all_results_df["dataset"]
results_df = preprocess_results_df(all_results_df, filter_dgp, dataset=dataset)
agg_df = get_agg_df(results_df, ["algorithm", "contamination", "epsilon", "inference"])
agg_df

In [ ]:
contamination_list = agg_df.index.get_level_values("contamination").unique().tolist()
nrows = 1
ncols = len(contamination_list)
trim_epsilon = 1.0
total_samples = 100
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(20,5), sharey=True)
fig.subplots_adjust(wspace=0.05)
pd.options.mode.chained_assignment = None  # default='warn'
trim_epsilon = 1.0
for j, contamination in enumerate(contamination_list):
    axis_df = agg_df.loc[:, contamination, :, :] #
    axis_df.loc[:, "is_pareto_front"] = is_maximise_pareto_front(axis_df["out_of_sample_var"].values, axis_df["out_of_sample_mean"].values)
    # for (algorithm, inference) in set(zip(axis_df.index.get_level_values("algorithm"), axis_df.index.get_level_values("inference"))):
    for (algorithm, inference) in [('dro_bas_mmd', "npl_mmd"), ('empirical_mmd', "empirical"), ('kl_dro_bas', "bayes"), ('kl_pp', "bayes"), ('kl_bdro', "bayes")]:
        df = axis_df.loc[algorithm, :trim_epsilon, inference]
        alpha = 1.0
        is_labelled=True
        mean_variance_plot(axes[j], df, **algorithm_inference_style(algorithm, inference, label_inference=False), offset=0.0, is_labelled=is_labelled, alpha=alpha, special_epsilons=[0.2])
    if j == 0:
        axes[j].legend(fontsize=14)
        axes[j].set_ylabel("Out-of-sample mean", fontsize=16)
        
    axes[j].set_title("$\eta$" + f"={contamination} with {NiceNameDGP[filter_dgp]}", fontsize=16)
    axes[j].set_xlabel("Out-of-sample variance", fontsize=16)
    axes[j].tick_params(axis='both', which='major', labelsize=12)
for ax in axes:
    ax.tick_params(axis='both', which='major', labelsize=16)
fig.tight_layout()
# plt.show()
fig.savefig(f"/Users/patrick/Experiments/misdro/2025_01_30_robas/{experiment_name}.pdf", bbox_inches="tight")

In [ ]:
idx = 10
dgp = experiment[idx]["dgp"]
num_observations = experiment[idx]["num_observations"]
contamination = experiment[idx]["contamination"]
dim = experiment[idx]["dim"]
replication = 0
generator = np.random.default_rng(replication)
data = sample_dgp(
    dgp, num_observations, dim=dim, contamination=contamination, generator=generator
)
dgp_mean = np.array([0.0, 1.5, -1.0, -3.0, 3.5])
dgp_mean_outl = dgp_mean + np.array([0.0, 50.0, 50.0, 50.0, 0.0])
dgp_cov = np.diag(np.array([1.0, 1.0, 1.0, 2.0, 3.0]))
print(json.dumps(experiment[idx], indent=4))

In [ ]:
# train a bayesian model with normal likelihood and NIW posterior
bayes_posterior = "normal_inverse_wishart"
theta_prior = default_prior_params(bayes_posterior, dim=dim)
theta_posterior = get_posterior_params(bayes_posterior, data, theta_prior)
mu_post, _, iota_post, Psi_post = theta_posterior
expected_Sigma = Psi_post / (iota_post - Psi_post.shape[0] - 1)

In [ ]:
# load one NPL posterior sample
posterior_df = pd.read_csv(npl_samples_dir / "npl_settings.csv").set_index(POSTERIOR_GB_COLS)
npl_params = {key: experiment[idx][key] for key in POSTERIOR_GB_COLS}
npl_uuid = get_npl_uuid(posterior_df, npl_params)
npl_uuid_dir = npl_samples_dir / npl_uuid
path_to_csv = npl_uuid_dir / f"npl_sample_{replication}.csv"
theta_sample = pd.read_csv(path_to_csv, index_col=False, header=None).values
i = 0
npl_mu = theta_sample[i,:dim]
vec_cov = theta_sample[i,dim:]
npl_cov = cholesky_param_to_covariance(dim, vec_cov)
npl_cov

In [ ]:
# plot the posterior against the DGP
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 10))
all_covs = np.concatenate([dgp_cov, expected_Sigma, np.cov(data.T), npl_cov])
vmax = np.max(all_covs)
vmin = np.min(all_covs)

axes[0][0].matshow(dgp_cov, vmin=vmin, vmax=vmax)
axes[0][0].set_title("DGP uncontaminated")

axes[0][1].matshow(np.cov(data.T), vmin=vmin, vmax=vmax)
axes[0][1].set_title("Contaminated empirical cov")

axes[1][0].matshow(npl_cov, vmin=vmin, vmax=vmax)
axes[1][0].set_title("NPL sample")

axes[1][1].matshow(expected_Sigma, vmin=vmin, vmax=vmax)
axes[1][1].set_title("Bayes posterior")

In [ ]:
# plot the posterior against the DGP
fig, axes = plt.subplots(ncols=2, figsize=(10, 5))
all_covs = np.concatenate([dgp_cov, npl_cov])
vmax = np.max(all_covs)
vmin = np.min(all_covs)

axes[0].matshow(dgp_cov)
axes[0].set_title("DGP uncontaminated")

axes[1].matshow(npl_cov)
axes[1].set_title("NPL sample")
